In [32]:
import pandas as pd
import re

In [89]:
DIRECTORY = 'qwen3-coder_480b'
FILE_PATH = f'{DIRECTORY}/sv_results_{DIRECTORY}_2.csv'

In [90]:
data = pd.read_csv(FILE_PATH)

In [91]:
data['iverilog_output'].value_counts()

iverilog_output
OK                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [92]:
def extract_code(output: str, iverilog: str)-> str:
    '''
    Auxiliar function to extract the systemverilog code from the LLM output.
    This is just a safeguard in case the model decides to putput something else than SystemVerilog code.
    '''
    is_valid = iverilog

    if iverilog == 'OK':
        has_testing_in_code = re.findall(r'\b(assert|property)\b', output)
        print(output, bool(has_testing_in_code))
        print('---------------------------------------------------------------')
        if not has_testing_in_code:
            is_valid = 'NON_VALID_OUTPUT'

    return is_valid

data['iverilog_output'] = data.apply(
    lambda row: extract_code(row['generated_code'], row['iverilog_output']), 
    axis=1
)

module seq_case71_assertions(
    input logic clk, reset,
    output logic [1:0] count
);

    property p_count_reset;
        @(posedge clk) (reset) |-> (count == 2'b00);
    endproperty
    assert property (p_count_reset);

    property p_count_00_to_01;
        @(posedge clk) (!reset && $past(count) == 2'b00) |-> (count == 2'b01);
    endproperty
    assert property (p_count_00_to_01);

    property p_count_01_to_10;
        @(posedge clk) (!reset && $past(count) == 2'b01) |-> (count == 2'b10);
    endproperty
    assert property (p_count_01_to_10);

    property p_count_10_to_11;
        @(posedge clk) (!reset && $past(count) == 2'b10) |-> (count == 2'b11);
    endproperty
    assert property (p_count_10_to_11);

    property p_count_11_to_00;
        @(posedge clk) (!reset && $past(count) == 2'b11) |-> (count == 2'b00);
    endproperty
    assert property (p_count_11_to_00);

    property p_count_default_to_00;
        @(posedge clk) (!reset && $isunknown($past(count))) |-> (count

In [93]:
data['iverilog_output'].value_counts()

iverilog_output
OK                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [94]:
data['iverilog_output'] = data['iverilog_output'].replace(['NO_SV_MODULE_FOUND', 'CODE_BLOCK_NOT_FOUND'], 'NON_VALID_OUTPUT')

In [95]:
data.to_csv(FILE_PATH, index=False)